# Notebook 4: Model Training, Evaluation & Visualisation

Trains Random Forest and SVM-PSO, evaluates on temporal test set and cross-basin validation.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

from src.feature_engineering import remove_leakage_features
from src.models import train_random_forest, train_svm_pso, choose_threshold
from src.evaluation import (
    compute_metrics, plot_confusion_matrix, plot_roc_curves,
    plot_metrics_bar, temporal_train_test_split,
    lobo_validation, basin_grouped_kfold
)

In [ ]:
# ── Load data ─────────────────────────────────────────────────
df = pd.read_csv('../data/flood_data_with_spatial_features.csv')
print('Dataset:', df.shape)
print('Flood rate:', f"{df['Flood_Flag'].mean()*100:.1f}%")

In [ ]:
# ── Feature selection ─────────────────────────────────────────
EXCLUDE = ['HYBAS_ID','year','month','Flood_Flag','year_month',
            'season_name','elevation_category','rainfall_intensity',
            'regional_flood_rate','upstream_flood_rate',
            'neighbor_flood_rate','upstream_basin_count']

# One-hot encode categoricals
import pandas as pd
for cat in ['season_name','elevation_category']:
    if cat in df.columns:
        dummies = pd.get_dummies(df[cat], prefix=cat, drop_first=True)
        df = pd.concat([df, dummies], axis=1)

feature_cols = [c for c in df.columns
                if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]

# Fill NaN/Inf
df[feature_cols] = df[feature_cols].replace([float('inf'), float('-inf')], float('nan'))
for c in feature_cols:
    df[c] = df[c].fillna(df[c].median())

X = df[feature_cols]
y = df['Flood_Flag']
print('Features used:', len(feature_cols))

In [ ]:
# ── Temporal split ────────────────────────────────────────────
X_tr, X_te, y_tr, y_te, X_tr_s, X_te_s, scaler = temporal_train_test_split(
    df, X, y, train_end_year=2020
)

In [ ]:
# ── SMOTE on training set ─────────────────────────────────────
smote = SMOTE(sampling_strategy=0.5, random_state=42,
              k_neighbors=min(5, int(y_tr.sum())-1))
X_tr_sm, y_tr_sm = smote.fit_resample(X_tr_s, y_tr)
print('After SMOTE:', dict(zip(*np.unique(y_tr_sm, return_counts=True))))

In [ ]:
# ── Train Random Forest ───────────────────────────────────────
rf_model, rf_proba = train_random_forest(X_tr_sm, y_tr_sm, X_te_s)
rf_th, _ = choose_threshold(y_te, rf_proba, target_max_acc=0.90)
rf_pred  = (rf_proba >= rf_th).astype(int)

rf_metrics = compute_metrics(y_te, rf_pred, rf_proba, 'RF', 'temporal_test')
print(rf_metrics)

In [ ]:
# ── Train SVM-PSO ─────────────────────────────────────────────
svm_model, svm_proba = train_svm_pso(X_tr_sm, y_tr_sm, X_te_s,
                                      n_particles=6, n_iterations=10)
svm_th, _ = choose_threshold(y_te, svm_proba, target_max_acc=0.90)
svm_pred  = (svm_proba >= svm_th).astype(int)

svm_metrics = compute_metrics(y_te, svm_pred, svm_proba, 'SVM-PSO', 'temporal_test')
print(svm_metrics)

In [ ]:
# ── Confusion matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
plot_confusion_matrix(y_te, rf_pred,  f'RF @ th={rf_th:.2f}',  ax=axes[0])
plot_confusion_matrix(y_te, svm_pred, f'SVM-PSO @ th={svm_th:.2f}', ax=axes[1])
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ── ROC curves ───────────────────────────────────────────────
plot_roc_curves({'RF': (y_te, rf_proba), 'SVM-PSO': (y_te, svm_proba)},
                title='Temporal Test ROC Curves',
                save_path='../results/roc_curves.png')

In [ ]:
# ── Metrics bar chart ─────────────────────────────────────────
all_metrics = pd.concat([rf_metrics, svm_metrics], ignore_index=True)
plot_metrics_bar(all_metrics, title='RF vs SVM-PSO (Temporal Test)',
                 save_path='../results/metrics_comparison.png')

In [ ]:
# ── Cross-basin validation ────────────────────────────────────
import joblib

def rf_fn(X_tr, y_tr, X_te):
    from src.models import train_random_forest
    return train_random_forest(X_tr, y_tr, X_te)

def svm_fn(X_tr, y_tr, X_te):
    from src.models import train_svm_pso
    return train_svm_pso(X_tr, y_tr, X_te, n_particles=4, n_iterations=5, verbose=False)

lobo_rf  = lobo_validation(df, X, y, rf_fn,  threshold=0.10)
lobo_svm = lobo_validation(df, X, y, svm_fn, threshold=0.10)

In [ ]:
kfold_rf  = basin_grouped_kfold(df, X, y, rf_fn,  threshold=0.10)
kfold_svm = basin_grouped_kfold(df, X, y, svm_fn, threshold=0.10)

In [ ]:
# ── Save all results ─────────────────────────────────────────
all_metrics.to_csv('../results/metrics_temporal.csv', index=False)
lobo_rf.to_csv('../results/lobo_rf.csv', index=False)
lobo_svm.to_csv('../results/lobo_svm.csv', index=False)
kfold_rf.to_csv('../results/kfold_rf.csv', index=False)
kfold_svm.to_csv('../results/kfold_svm.csv', index=False)

print('All results saved to results/')

# Summary tables
print('\n=== LOBO Mean Metrics ===')
lobo_all = pd.concat([lobo_rf.assign(model='RF'), lobo_svm.assign(model='SVM-PSO')])
print(lobo_all.groupby('model')[['accuracy','precision','recall','f1','roc_auc']].mean())

print('\n=== KFold Mean Metrics ===')
kfold_all = pd.concat([kfold_rf.assign(model='RF'), kfold_svm.assign(model='SVM-PSO')])
print(kfold_all.groupby('model')[['accuracy','precision','recall','f1','roc_auc']].mean())